<a href="https://colab.research.google.com/github/noone878/DataScience_230401010262_SEPTIAN-AL-RIZKI/blob/main/Pertemuan12_SEPTIAN_AL_RIZKI_230401010262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistem Rekomendasi Hibrida: Analisis Pola Belanja & Kategori
Notebook ini mendemonstrasikan cara menggabungkan **Market Basket Analysis** (Association Rules) dengan **Content-Based Filtering** untuk membangun sistem rekomendasi yang cerdas.

In [31]:
import warnings
import os

# Mematikan semua peringatan Python termasuk DeprecationWarning dari library eksternal
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore')

# Mengatur variabel lingkungan untuk mematikan peringatan di tingkat sistem
os.environ['PYTHONWARNINGS'] = 'ignore'

print('Filter peringatan telah diaktifkan.')

Filter peringatan telah diaktifkan.


### 1. Konfigurasi Lingkungan
Kita memulainya dengan mematikan pesan *warning* agar output analisis tetap bersih dan profesional.

In [32]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Konfigurasi Seed untuk reproduksibilitas
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Simulasi Data Transaksi
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan Pola Bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print(f'Total Transaksi: {len(transaksi)}')
print('3 Sampel Transaksi Pertama:')
for t in transaksi[:3]:
    print(f'- {t}')

Total Transaksi: 50
3 Sampel Transaksi Pertama:
- [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
- [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
- [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]


### 2. Simulasi Data Transaksi
Di sini kita membuat dataset belanja buatan (dummy) sebanyak 50 transaksi.
*   **Produk**: Daftar kebutuhan pokok (Roti, Susu, dll).
*   **Pola Bisnis**: Kita menyuntikkan logika khusus di mana setiap pembelian 'Roti' kemungkinan besar dibarengi dengan 'Selai'. Ini mensimulasikan kebiasaan nyata konsumen.

In [33]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Konfigurasi Seed untuk reproduksibilitas
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Simulasi Data Transaksi
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan Pola Bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print(f'Total Transaksi: {len(transaksi)}')
print('3 Sampel Transaksi Pertama:')
for t in transaksi[:3]:
    print(f'- {t}')

Total Transaksi: 50
3 Sampel Transaksi Pertama:
- [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
- [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
- [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]


In [34]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Konfigurasi Seed untuk reproduksibilitas
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Simulasi Data Transaksi
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan Pola Bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print(f'Total Transaksi: {len(transaksi)}')
print('3 Sampel Transaksi Pertama:')
for t in transaksi[:3]:
    print(f'- {t}')

Total Transaksi: 50
3 Sampel Transaksi Pertama:
- [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
- [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
- [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]


In [35]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Konfigurasi Seed untuk reproduksibilitas
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Simulasi Data Transaksi
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan Pola Bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print(f'Total Transaksi: {len(transaksi)}')
print('3 Sampel Transaksi Pertama:')
for t in transaksi[:3]:
    print(f'- {t}')

Total Transaksi: 50
3 Sampel Transaksi Pertama:
- [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
- [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
- [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]


In [36]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Konfigurasi Seed untuk reproduksibilitas
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Simulasi Data Transaksi
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan Pola Bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print(f'Total Transaksi: {len(transaksi)}')
print('3 Sampel Transaksi Pertama:')
for t in transaksi[:3]:
    print(f'- {t}')

Total Transaksi: 50
3 Sampel Transaksi Pertama:
- [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
- [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
- [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]


In [37]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

# Encoding data transaksi ke format matriks boolean
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("Preview Matriks Transaksi (5 baris pertama):")
display(df.head())

# Mencari Frequent Itemsets
# min_support 0.1 berarti produk muncul di minimal 10% total transaksi
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print("\n10 Itemset Paling Sering Muncul:")
display(freq_items.head(10))

Preview Matriks Transaksi (5 baris pertama):


,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False



10 Itemset Paling Sering Muncul:


,support,itemsets
5,0.52,(Selai)
8,0.46,(Teh)
3,0.42,(Mentega)
9,0.36,(Telur)
1,0.34,(Keju)
0,0.32,(Gula)
2,0.32,(Kopi)
4,0.32,(Roti)
7,0.32,(Susu)
36,0.24,"(Teh, Selai)"


### 3. Persiapan Data & Analisis Itemset
Kode ini mengubah data teks transaksi menjadi matriks boolean (True/False).
*   **Support**: Mengukur seberapa sering sebuah kombinasi produk muncul.
*   Kita menyaring produk yang muncul setidaknya di 10% transaksi (`min_support=0.1`).

In [38]:
from mlxtend.frequent_patterns import association_rules

# Membuat Aturan Asosiasi
# Metrik Confidence > 0.5 berarti probabilitas konsekuen dibeli jika anteseden dibeli > 50%
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)

# Filter: Lift > 1 menunjukkan korelasi positif antar produk
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("Top 10 Aturan Asosiasi Terkuat (Berdasarkan Lift):")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Top 10 Aturan Asosiasi Terkuat (Berdasarkan Lift):


,antecedents,consequents,support,confidence,lift
9,"(Teh, Keju)",(Telur),0.12,0.857143,2.380952
15,"(Mentega, Selai)",(Kopi),0.10,0.625000,1.953125
11,"(Gula, Roti)",(Selai),0.10,1.000000,1.923077
7,(Sereal),(Mentega),0.14,0.777778,1.851852
8,"(Teh, Telur)",(Keju),0.12,0.600000,1.764706
13,"(Kopi, Selai)",(Mentega),0.10,0.714286,1.700680
10,"(Keju, Telur)",(Teh),0.12,0.750000,1.630435
12,"(Gula, Selai)",(Roti),0.10,0.500000,1.562500
14,"(Kopi, Mentega)",(Selai),0.10,0.714286,1.373626
1,(Roti),(Selai),0.22,0.687500,1.322115


### 4. Menghasilkan Aturan Asosiasi (Association Rules)
Langkah ini mencari hubungan sebab-akibat antar produk.
*   **Confidence**: Keyakinan bahwa jika membeli produk A, maka akan membeli produk B.
*   **Lift**: Menunjukkan kekuatan hubungan. Lift > 1 berarti produk tersebut saling tarik-menarik lebih dari sekadar kebetulan.

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

# Dataset Katalog Produk berdasarkan Kategori
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

# Menghitung Similarity berdasarkan Kategori (Content-Based)
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    if nama_produk not in katalog['produk'].values:
        return []

    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    # Urutkan berdasarkan kemiripan tertinggi, kecualikan diri sendiri
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    res_idx = [s[0] for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[res_idx]['produk'].tolist()

print(f"Rekomendasi Kategori Serupa dengan 'Roti': {rekomendasi_serupa('Roti')}")

Rekomendasi Kategori Serupa dengan 'Roti': ['Selai', 'Sereal', 'Susu']


### 5. Content-Based Filtering (Kemiripan Kategori)
Berbeda dengan pola belanja, metode ini melihat karakteristik produk. Jika Anda menyukai satu produk 'Bakery', sistem akan merekomendasikan produk lain dari kategori 'Bakery' menggunakan perhitungan *Cosine Similarity*.

In [40]:
produk_target = 'Roti'

print(f"=== SISTEM REKOMENDASI UNTUK: {produk_target.upper()} ===\n")

# 1. Rekomendasi Berbasis Perilaku Belanja (Association Rules)
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]
print('1. Rekomendasi Cross-Sell (User juga membeli):')
if not rules_terkait.empty:
    display(rules_terkait[['consequents', 'confidence', 'lift']].head())
else:
    print("   Tidak ditemukan pola belanja yang signifikan.")

# 2. Rekomendasi Berbasis Karakteristik Produk (Content-Based)
print('\n2. Rekomendasi Produk Serupa (Kategori yang sama):')
print(f"   {rekomendasi_serupa(produk_target)}")

print("\nKesimpulan: Association Rules memberikan insight pelengkap (misal: Roti -> Selai), "\
      "sedangkan Content-Based memberikan alternatif produk sejenis (misal: Roti -> Sereal).")

=== SISTEM REKOMENDASI UNTUK: ROTI ===

1. Rekomendasi Cross-Sell (User juga membeli):


,consequents,confidence,lift
11,(Selai),1.0000,1.923077
1,(Selai),0.6875,1.322115



2. Rekomendasi Produk Serupa (Kategori yang sama):
   ['Selai', 'Sereal', 'Susu']

Kesimpulan: Association Rules memberikan insight pelengkap (misal: Roti -> Selai), sedangkan Content-Based memberikan alternatif produk sejenis (misal: Roti -> Sereal).


### 6. Hasil Akhir & Perbandingan
Ini adalah bagian terpenting di mana kita melihat dua perspektif rekomendasi untuk 'Roti':
1.  **Rekomendasi Cross-Sell**: Menyarankan produk pelengkap (Roti -> Selai).
2.  **Rekomendasi Produk Serupa**: Menyarankan produk sejenis berdasarkan kategori.

Strategi ini memungkinkan toko untuk meningkatkan penjualan tambahan sekaligus memberikan alternatif jika stok produk utama habis.